In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

# -----------------------------
# Interactive demo (hardcoded derivatives/integrals)
# -----------------------------
def lti_demo(signal_type="sin", tau=1.0, omega=1.0, a=1.0):
    # causal time axis
    t = np.linspace(0,10,2000)

    # -----------------------------
    # Input signals & hardcoded derivatives/integrals
    # -----------------------------
    if signal_type == "sin":
        x = np.sin(omega*t)
        y_diff = omega*np.cos(omega*t)               # dx/dt
        y_int  = 1/omega*(1 - np.cos(omega*t))       # ∫_0^t x(τ)dτ
        diff_title = r"Differentiator output: $y(t)=dx/dt = \omega \cos(\omega t)$"
        int_title  = r"Integrator output: $y(t)=\int_0^t x(\tau)d\tau = \frac{1-\cos(\omega t)}{\omega}$"
    elif signal_type == "square":
        x = np.sign(np.sin(omega*t))
        # Derivative: spikes at zero crossings
        y_diff = np.zeros_like(x)
        zero_crossings = np.where(np.diff(np.sign(x)))[0]
        y_diff[zero_crossings] = 2*omega
        # Integral: numeric approximation
        y_int = np.zeros_like(x)
        y_int[1:] = np.cumsum(x[:-1]*(t[1]-t[0]))
        diff_title = r"Differentiator output: spikes at zero crossings (Spikes represent Dirac delta at jump points)"
        int_title  = r"Integrator output: cumulative sum (∫_0^t x(τ)dτ)"
    elif signal_type == "exp":
        x = np.exp(-a*t)
        y_diff = -a*np.exp(-a*t)                     # dx/dt
        y_int  = (1 - np.exp(-a*t))/a               # ∫_0^t x(τ)dτ
        diff_title = r"Differentiator output: $y(t)=dx/dt = -a e^{-at}$"
        int_title  = r"Integrator output: $y(t)=\int_0^t x(\tau)d\tau = \frac{1-e^{-at}}{a}$"

    # -----------------------------
    # Time delay (causal)
    # -----------------------------
    y_delay = np.zeros_like(x)
    idx_delay = t >= tau
    y_delay[idx_delay] = np.interp(t[idx_delay] - tau, t, x, left=0)
    delay_title = r"Time delay output: $y(t)=x(t-\tau)$, $\tau={:.2f}$".format(tau)

    # -----------------------------
    # Plot
    # -----------------------------
    fig, axes = plt.subplots(4,1,figsize=(10,10), sharex=True)

    # Input signal
    axes[0].plot(t, x, color='blue')
    axes[0].set_title("Input signal: x(t)")
    axes[0].grid()
    axes[0].set_xlim(t[0], t[-1])
    axes[0].set_ylim(np.min(x)*1.2, np.max(x)*1.2)

    # Differentiator
    axes[1].plot(t, y_diff, color='green')
    axes[1].set_title(diff_title)
    axes[1].grid()
    axes[1].set_ylim(np.min(y_diff)*1.2, np.max(y_diff)*1.2)

    # Integrator
    axes[2].plot(t, y_int, color='orange')
    axes[2].set_title(int_title)
    axes[2].grid()
    axes[2].set_ylim(np.min(y_int)*1.2, np.max(y_int)*1.2)

    # Time delay
    axes[3].plot(t[idx_delay], y_delay[idx_delay], color='red', label=f"x(t-τ)")
    axes[3].plot(t, x, color='blue', alpha=0.3, label="x(t)")
    axes[3].set_title(delay_title)
    axes[3].legend()
    axes[3].grid()
    axes[3].set_ylim(np.min(x)*1.2, np.max(x)*1.2)

    plt.tight_layout()
    plt.show()

# -----------------------------
# Widgets
# -----------------------------
interact(
    lti_demo,
    signal_type=["sin","square","exp"],
    tau=(0.0,3.0,0.1),
    omega=(0.5,5.0,0.1),
    a=(0.1,3.0,0.1)
)